# Stage 1: Data Inspection & Raw PGN Audit

**Project**: EDA-Driven Behavioral Feature Engineering for Human–AI Chess Move Detection  
**Goal**: Inspect all raw chess data files in `data/raw/`, audit game formats, header metadata, player labels, clock comments, and identify cleaning requirements without modifying raw files.


## 1. Environment & Setup
Load necessary libraries for chess parsing and tabular analysis.

In [ ]:
import os
import glob
import re
from collections import Counter, defaultdict
import chess
import chess.pgn
import pandas as pd
import numpy as np

RAW_DIR = os.path.join("..", "data", "raw")
print("Raw directory:", os.path.abspath(RAW_DIR))


## 2. Raw Files Inventory
Inspect all `.pgn` files, their disk sizes, and header game counts.

In [ ]:
pgn_files = sorted(glob.glob(os.path.join(RAW_DIR, "*.pgn")))
file_inventory = []

for fpath in pgn_files:
    fname = os.path.basename(fpath)
    fsize = os.path.getsize(fpath)
    g_count = 0
    with open(fpath, "r", encoding="utf-8", errors="replace") as f:
        while True:
            headers = chess.pgn.read_headers(f)
            if headers is None: break
            g_count += 1
    file_inventory.append({
        "file_name": fname,
        "size_bytes": fsize,
        "size_mb": round(fsize / (1024 * 1024), 2),
        "raw_games": g_count
    })

df_inv = pd.DataFrame(file_inventory)
display(df_inv)
print("Total raw game instances across all files:", df_inv["raw_games"].sum())


## 3. Game Deduplication Analysis
Check for duplicate game instances across files by tracking the unique 8-character Lichess identifier from `Site`.

In [ ]:
game_registry = defaultdict(list)

for fpath in pgn_files:
    fname = os.path.basename(fpath)
    with open(fpath, "r", encoding="utf-8", errors="replace") as f:
        while True:
            headers = chess.pgn.read_headers(f)
            if headers is None: break
            site = headers.get("Site", "")
            gid = site.split("/")[-1] if "lichess.org" in site else site
            game_registry[gid].append(fname)

total_games = sum(len(v) for v in game_registry.values())
unique_games = len(game_registry)
duplicates = {k: v for k, v in game_registry.items() if len(v) > 1}

print(f"Total game records: {total_games}")
print(f"Unique games: {unique_games}")
print(f"Duplicate instances: {len(duplicates)}")

dup_patterns = Counter(tuple(v) for v in duplicates.values())
for pair, count in dup_patterns.items():
    print(f"  {count} games appear in both: {pair}")


## 4. Header Metadata & Available Information
Analyze presence of standard header tags across unique games.

In [ ]:
seen_ids = set()
tag_counts = Counter()
white_titles = Counter()
black_titles = Counter()
tc_counts = Counter()
ratings_w = []
ratings_b = []

for fpath in pgn_files:
    with open(fpath, "r", encoding="utf-8", errors="replace") as f:
        while True:
            headers = chess.pgn.read_headers(f)
            if headers is None: break
            site = headers.get("Site", "")
            gid = site.split("/")[-1] if "lichess.org" in site else site
            if gid in seen_ids: continue
            seen_ids.add(gid)
            
            for k in headers.keys():
                tag_counts[k] += 1
            white_titles[headers.get("WhiteTitle", "None")] += 1
            black_titles[headers.get("BlackTitle", "None")] += 1
            tc_counts[headers.get("TimeControl", "Missing")] += 1
            try:
                ratings_w.append(int(headers.get("WhiteElo", 0)))
                ratings_b.append(int(headers.get("BlackElo", 0)))
            except:
                pass

print("Top 10 Header Tags Presence:")
for k, v in tag_counts.most_common(10):
    print(f"  {k:15s}: {v:5d} ({v/len(seen_ids)*100:.1f}%)")

print("\nWhite Title Distribution:", white_titles)
print("Black Title Distribution:", black_titles)
print("\nTop 5 Time Controls:", tc_counts.most_common(5))


## 5. Inspection of Clock Annotations & Move Times
Parse representative games to inspect `[%clk H:MM:SS]` comments and test move-time calculation.

In [ ]:
sample_game_path = os.path.join(RAW_DIR, "lichess_human_vs_bot_games.pgn")
with open(sample_game_path, "r", encoding="utf-8") as f:
    game = chess.pgn.read_game(f)

print("Sample Game Headers:")
print(f"  Event: {game.headers.get('Event')} | TimeControl: {game.headers.get('TimeControl')}")
print(f"  White: {game.headers.get('White')} ({game.headers.get('WhiteTitle', 'Human')})")
print(f"  Black: {game.headers.get('Black')} ({game.headers.get('BlackTitle', 'Human')})")

board = game.board()
tc = game.headers.get("TimeControl", "180+2")
base_s, inc_s = map(int, tc.split("+"))
w_prev = base_s
b_prev = base_s

print("\nFirst 8 Moves Analysis:")
for idx, node in enumerate(list(game.mainline())[:8]):
    is_white = (idx % 2 == 0)
    color = "White" if is_white else "Black"
    move_num = (idx // 2) + 1
    clk = node.clock()
    prev = w_prev if is_white else b_prev
    
    if idx < 2:
        dt = max(0.0, prev - clk) if clk is not None else None
    else:
        dt = prev + inc_s - clk if clk is not None else None
        
    if is_white: w_prev = clk
    else: b_prev = clk
    
    san = board.san(node.move)
    board.push(node.move)
    print(f"  Move {move_num:2d}{'.' if is_white else '...'} [{color:5s}]: {san:6s} | clk={clk}s | thinking_time={dt}s")


## 6. Findings & Preprocessing Strategy Summary
1. **Deduplication**: 3,021 duplicate game blocks identified across files; deduplicate strictly on unique `game_id`.
2. **Empty Games**: 12 games have 0 moves (aborted/forfeited before move 1); must be excluded.
3. **Untimed Games**: 8 correspondence games have no clocks (`TimeControl: "-"`); must be excluded.
4. **Clock Completeness**: 100% of moves in valid timed games have `[%clk]` annotations.
5. **Negative Clock Inversions**: 41 moves have negative deltas due to +15s opponent gifts or lag compensation; set `move_time = NaN` without fabrication.
6. **Labels**: Platform `BOT` title provides 100% verified ground truth.